In [ ]:
"""
Problem Set 4, Question 3 — Almgren-Chriss optimal execution.

Solve continuous-time optimal execution under quadratic transient (temporary)
impact via HJB + ansatz + Riccati ODE, and verify the closed-form trajectory
matches the discrete-time Almgren-Chriss (2001) result in the tau -> 0 limit.

Convention: x_t is REMAINING inventory (starts at x0, must hit 0 at T).
v_t = -dx/dt is the trading (liquidation) rate.

Cost functional (standard Almgren-Chriss convention):
    J(v) = E[ integral_0^T ( eta * v_t^2 + lambda * sigma^2 * x_t^2 ) dt ]

    NOTE: the risk-penalty term is lambda*sigma^2*x_t^2 -- NOT (lambda/2)*sigma^2*x_t^2.
    An earlier draft of this module used a 1/2 prefactor on the risk term,
    which is a common, easy-to-introduce inconsistency: it still gives a
    closed-form sinh/coth solution, but with kappa = sqrt(lambda*sigma^2/(2*eta))
    instead of the standard kappa = sqrt(lambda*sigma^2/eta), and that
    kappa does NOT match the tau->0 limit of the discrete-time AC formula
    cosh(kappa_tilde*tau) = 1 + tau^2*lambda*sigma^2/(2*eta), whose limit is
    sqrt(lambda*sigma^2/eta). Keeping the conventions aligned (no 1/2 on the
    risk term) is what makes the discrete -> continuous limit check in
    Part 3 below actually converge. Verified symbolically with sympy.

    eta    : temporary impact coefficient (cost of trading fast)
    lambda : risk-aversion weight on holding inventory
    sigma  : volatility of the asset (risk per unit time per unit inventory)

This is deterministic-control once you note that under the standard AC setup
(no feedback of the price shocks into the optimal trajectory itself), the
optimal x_t path is the same path that minimizes the deterministic functional
above; sigma enters only through the risk-penalty term, not through a
stochastic state equation for x_t. This is exactly why the HJB ansatz
V(t,x) = 0.5*h(t)*x^2 closes into a deterministic Riccati ODE for h(t).
"""

import math
import torch
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------
# 1. Continuous-time closed-form solution
# ----------------------------------------------------------------------

def kappa_continuous(eta: float, lam: float, sigma: float) -> float:
    """
    Urgency parameter for the continuous-time problem.

        kappa = sqrt( lambda * sigma^2 / eta )

    Derivation: plug the ansatz h(t) = C*coth(kappa*(T-t)) into the Riccati
    ODE  h'(t) - h(t)^2/(2*eta) + 2*lambda*sigma^2 = 0  (this is the HJB
    reduction under running cost eta*v^2 + lambda*sigma^2*x^2, i.e. NO 1/2
    prefactor on the risk term -- see module docstring) and match
    coefficients of sinh^2(kappa*(T-t)) and the constant term separately:
        C^2 = 4*eta*lambda*sigma^2          (coefficient of sinh^2 term)
        kappa = C/(2*eta)                    (constant term)
    Combining: kappa = sqrt(lambda*sigma^2/eta). Confirmed symbolically
    (sympy) that this is the unique kappa making the HJB residual vanish
    identically in t, AND that it matches the small-tau limit of the
    discrete kappa_tilde formula below (see kappa_tilde_discrete).
    """
    return math.sqrt(lam * sigma**2 / eta)


def x_continuous(t, x0: float, T: float, eta: float, lam: float, sigma: float):
    """
    Closed-form optimal remaining-inventory trajectory.

        x(t) = x0 * sinh(kappa * (T - t)) / sinh(kappa * T)

    t may be a torch.Tensor (vectorized) or a python float.
    """
    kappa = kappa_continuous(eta, lam, sigma)
    t = torch.as_tensor(t, dtype=torch.float64)
    num = torch.sinh(kappa * (T - t))
    den = math.sinh(kappa * T)
    return x0 * num / den


def v_continuous(t, x0: float, T: float, eta: float, lam: float, sigma: float):
    """
    Optimal trading rate v(t) = -dx/dt, obtained here via autograd applied
    to x_continuous (rather than re-deriving the closed form by hand) --
    consistent with the course convention of using PyTorch autograd for
    all derivatives.

    Note: we clone the input before calling requires_grad_, because
    torch.as_tensor on an already-float64 tensor returns the SAME
    underlying object (no copy) -- calling requires_grad_(True) on it
    would mutate the caller's tensor in place as a side effect.
    """
    t = torch.as_tensor(t, dtype=torch.float64).clone().requires_grad_(True)
    x = x_continuous(t, x0, T, eta, lam, sigma)
    grad = torch.autograd.grad(x.sum(), t, create_graph=False)[0]
    return -grad


def h_continuous(t, T: float, eta: float, lam: float, sigma: float):
    """
    Riccati solution h(t) = 2*sqrt(eta*lambda*sigma^2) * coth(kappa*(T-t))
    i.e. the value-function curvature V(t,x) = 0.5 * h(t) * x^2, consistent
    with running cost eta*v^2 + lambda*sigma^2*x^2 (no 1/2 on risk term).
    """
    kappa = kappa_continuous(eta, lam, sigma)
    t = torch.as_tensor(t, dtype=torch.float64)
    coth = torch.cosh(kappa * (T - t)) / torch.sinh(kappa * (T - t))
    return 2.0 * math.sqrt(eta * lam * sigma**2) * coth


# ----------------------------------------------------------------------
# 2. Discrete-time Almgren-Chriss (2001) solution
# ----------------------------------------------------------------------

def kappa_tilde_discrete(tau: float, eta: float, lam: float, sigma: float) -> float:
    """
    Discrete urgency parameter kappa_tilde defined implicitly by

        cosh(kappa_tilde * tau) = 1 + tau^2 * lambda * sigma^2 / (2*eta)

    Solve via arccosh of the RHS. Small-tau Taylor expansion of cosh gives
    1 + (kappa_tilde*tau)^2/2 ~= 1 + tau^2*lambda*sigma^2/(2*eta), so
    kappa_tilde -> sqrt(lambda*sigma^2/eta) as tau -> 0 -- matching
    kappa_continuous above exactly. This is the limit check in Part 4.
    """
    rhs = 1.0 + (tau**2 * lam * sigma**2) / (2.0 * eta)
    return math.acosh(rhs) / tau


def x_discrete(j_array, x0: float, N: int, T: float, eta: float, lam: float, sigma: float):
    """
    Discrete optimal remaining-inventory trajectory at step j = 0..N:

        x_j = x0 * sinh(kappa_tilde * (T - t_j)) / sinh(kappa_tilde * T)

    where t_j = j * tau, tau = T / N. j_array is a 1D array/tensor of step
    indices.
    """
    tau = T / N
    kappa_t = kappa_tilde_discrete(tau, eta, lam, sigma)
    j_array = torch.as_tensor(j_array, dtype=torch.float64)
    t_j = j_array * tau
    num = torch.sinh(kappa_t * (T - t_j))
    den = math.sinh(kappa_t * T)
    return x0 * num / den


# ----------------------------------------------------------------------
# 3. HJB residual check (autograd) -- verification theorem in practice
# ----------------------------------------------------------------------

def hjb_residual(t, x, T: float, eta: float, lam: float, sigma: float):
    """
    Evaluate the HJB residual

        R(t,x) = V_t(t,x) - V_x(t,x)^2/(4*eta) + lambda*sigma^2*x^2

    for the candidate value function V(t,x) = 0.5*h(t)*x^2, using autograd
    for V_t and V_x rather than hand-differentiating h(t).

    Should be ~machine-zero for the true h(t); we also perturb h(t) (e.g.
    scale it) and show the residual blows up, demonstrating the solution is
    *not* generic -- it is pinned down by the Riccati ODE.
    """
    t = torch.as_tensor(t, dtype=torch.float64).clone().requires_grad_(True)
    x = torch.as_tensor(x, dtype=torch.float64).clone().requires_grad_(True)

    h = h_continuous(t, T, eta, lam, sigma)
    V = 0.5 * h * x**2

    V_t = torch.autograd.grad(V.sum(), t, create_graph=True, retain_graph=True)[0]
    V_x = torch.autograd.grad(V.sum(), x, create_graph=True, retain_graph=True)[0]

    R = V_t - V_x**2 / (4 * eta) + lam * sigma**2 * x**2
    return R.detach()


def hjb_residual_perturbed(t, x, T, eta, lam, sigma, scale: float):
    """Same residual but with h(t) artificially scaled by `scale` != 1,
    breaking the Riccati equation on purpose."""
    t = torch.as_tensor(t, dtype=torch.float64).clone().requires_grad_(True)
    x = torch.as_tensor(x, dtype=torch.float64).clone().requires_grad_(True)

    h = scale * h_continuous(t, T, eta, lam, sigma)
    V = 0.5 * h * x**2

    V_t = torch.autograd.grad(V.sum(), t, create_graph=True, retain_graph=True)[0]
    V_x = torch.autograd.grad(V.sum(), x, create_graph=True, retain_graph=True)[0]

    R = V_t - V_x**2 / (4 * eta) + lam * sigma**2 * x**2
    return R.detach()


# ----------------------------------------------------------------------
# 4. Main experiment: limit verification + plots/CSV
# ----------------------------------------------------------------------

def main():
    torch.manual_seed(0)

    # Parameters (illustrative; swap in calibrated values if your starter
    # repo provides a fitted eta/lambda from Coincall fill data)
    x0 = 1.0       # normalized starting inventory (e.g. 1 BTC-PERP unit)
    T = 1.0        # liquidation horizon (1 trading day, normalized)
    eta = 0.05     # temporary impact coefficient
    lam = 0.02     # risk-aversion weight
    sigma = 2.0    # volatility (USD / sqrt(time), matching Week 3's units)

    kappa = kappa_continuous(eta, lam, sigma)
    print(f"[setup] kappa (continuous) = {kappa:.6f}")

    # --- (a) Plot continuous trajectory ---
    t_grid = torch.linspace(0, T, 400, dtype=torch.float64)
    x_cont = x_continuous(t_grid, x0, T, eta, lam, sigma).detach()
    v_cont = v_continuous(t_grid, x0, T, eta, lam, sigma).detach()

    # --- (b) Discrete trajectories for increasing N ---
    N_list = [5, 10, 50, 200, 1000]
    discrete_results = {}
    for N in N_list:
        j = torch.arange(0, N + 1, dtype=torch.float64)
        x_disc = x_discrete(j, x0, N, T, eta, lam, sigma)
        t_disc = j * (T / N)
        discrete_results[N] = (t_disc, x_disc)

    # --- (c) Limit check: sup-norm error between discrete and continuous,
    #     evaluated at the discrete grid points, as N -> infinity ---
    errors = []
    for N in N_list:
        t_disc, x_disc = discrete_results[N]
        x_cont_at_disc = x_continuous(t_disc, x0, T, eta, lam, sigma)
        err = (x_disc - x_cont_at_disc).abs().max().item()
        tau = T / N
        errors.append({"N": N, "tau": tau, "max_abs_error": err})

    err_df = pd.DataFrame(errors)
    err_df["error_over_tau"] = err_df["max_abs_error"] / err_df["tau"]
    err_df["error_over_tau2"] = err_df["max_abs_error"] / err_df["tau"] ** 2
    err_df.to_csv("ac_limit_convergence.csv", index=False)

    print("\n[limit check] discrete -> continuous as N -> infinity")
    print(err_df.to_string(index=False))

    # Confirm error decreases monotonically and ratio error/tau^2 stabilizes
    # (indicating O(tau^2) convergence, consistent with cosh(.) being even
    # in tau so the leading correction is quadratic)
    decreasing = all(
        err_df["max_abs_error"].iloc[i] > err_df["max_abs_error"].iloc[i + 1]
        for i in range(len(err_df) - 1)
    )
    print(f"\n[check] error strictly decreasing in N: {decreasing}")
    print(f"[check] error/tau^2 stabilizing (approx constant)? values: "
          f"{err_df['error_over_tau2'].round(4).tolist()}")

    # --- (d) Trajectory comparison CSV (for the write-up figure) ---
    traj_rows = []
    for N in N_list:
        t_disc, x_disc = discrete_results[N]
        for tt, xx in zip(t_disc.tolist(), x_disc.tolist()):
            traj_rows.append({"N": N, "t": tt, "x_discrete": xx})
    traj_df = pd.DataFrame(traj_rows)

    cont_df = pd.DataFrame({
        "t": t_grid.tolist(),
        "x_continuous": x_cont.tolist(),
        "v_continuous": v_cont.tolist(),
    })
    cont_df.to_csv("ac_continuous_trajectory.csv", index=False)
    traj_df.to_csv("ac_discrete_trajectories.csv", index=False)

    # --- (e) HJB residual check ---
    t_test = torch.tensor([0.1, 0.3, 0.5, 0.7, 0.9], dtype=torch.float64)
    x_test = torch.tensor([0.8, 0.6, 0.5, 0.3, 0.1], dtype=torch.float64)

    R_true = hjb_residual(t_test, x_test, T, eta, lam, sigma)
    R_perturbed = hjb_residual_perturbed(t_test, x_test, T, eta, lam, sigma, scale=1.2)

    print("\n[HJB residual] true h(t): max|R| = "
          f"{R_true.abs().max().item():.3e}  (should be ~machine-zero)")
    print("[HJB residual] perturbed h(t) (scale=1.2): max|R| = "
          f"{R_perturbed.abs().max().item():.3e}  (should be large)")

    # --- (f) Plot: trajectories ---
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    ax = axes[0]
    ax.plot(t_grid.numpy(), x_cont.numpy(), "k-", lw=2.5, zorder=5,
            label="continuous (closed form)")
    # Only show the coarsest few N -- finer N are visually indistinguishable
    # from the continuous curve at this scale (that IS the point, but a
    # dedicated log-log error panel on the right makes that point better).
    N_to_plot = [5, 10, 50]
    colors = plt.cm.plasma([0.15, 0.5, 0.8])
    for N, c in zip(N_to_plot, colors):
        t_disc, x_disc = discrete_results[N]
        ax.plot(t_disc.numpy(), x_disc.numpy(), "o--", color=c, ms=5,
                lw=1.2, label=f"discrete N={N}")
    ax.set_xlabel("t")
    ax.set_ylabel("remaining inventory x(t)")
    ax.set_title("Optimal liquidation trajectory:\ncoarse discrete grids vs. continuous limit")
    ax.legend(fontsize=8)

    ax2 = axes[1]
    ax2.loglog(err_df["tau"], err_df["max_abs_error"], "o-", color="firebrick",
               label="max abs error")
    # reference O(tau^2) line for visual comparison
    tau_ref = err_df["tau"].values
    c_ref = err_df["max_abs_error"].iloc[-1] / tau_ref[-1] ** 2
    ax2.loglog(tau_ref, c_ref * tau_ref ** 2, "k--", lw=1, label=r"$O(\tau^2)$ reference")
    ax2.set_xlabel(r"$\tau = T/N$")
    ax2.set_ylabel("max |x_discrete - x_continuous|")
    ax2.set_title("Convergence rate (log-log)")
    ax2.legend(fontsize=8)
    ax2.invert_xaxis()

    fig.tight_layout()
    fig.savefig("ac_convergence.png", dpi=150)
    print("\n[output] wrote ac_limit_convergence.csv, "
          "ac_continuous_trajectory.csv, ac_discrete_trajectories.csv, "
          "ac_convergence.png")


if __name__ == "__main__":
    main()

[setup] kappa (continuous) = 1.264911

[limit check] discrete -> continuous as N -> infinity
   N   tau  max_abs_error  error_over_tau  error_over_tau2
   5 0.200   4.045862e-04        0.002023         0.010115
  10 0.100   1.017365e-04        0.001017         0.010174
  50 0.020   4.077084e-06        0.000204         0.010193
 200 0.005   2.548367e-07        0.000051         0.010193
1000 0.001   1.019141e-08        0.000010         0.010191

[check] error strictly decreasing in N: True
[check] error/tau^2 stabilizing (approx constant)? values: [0.0101, 0.0102, 0.0102, 0.0102, 0.0102]

[HJB residual] true h(t): max|R| = 2.580e-17  (should be ~machine-zero)
[HJB residual] perturbed h(t) (scale=1.2): max|R| = 2.879e-02  (should be large)

[output] wrote ac_limit_convergence.csv, ac_continuous_trajectory.csv, ac_discrete_trajectories.csv, ac_convergence.png


In [ ]:
"""
pytest suite for Problem Set 4, Question 3 (Almgren-Chriss optimal execution).

Run with: pytest -v test_ac_execution.py
"""

import math
import torch
import pytest

from ac_execution import (
    kappa_continuous,
    x_continuous,
    v_continuous,
    h_continuous,
    kappa_tilde_discrete,
    x_discrete,
    hjb_residual,
    hjb_residual_perturbed,
)

ETA, LAM, SIGMA, T, X0 = 0.05, 0.02, 2.0, 1.0, 1.0


def test_kappa_continuous_matches_discrete_limit():
    """kappa_tilde_discrete(tau) -> kappa_continuous as tau -> 0."""
    kappa_c = kappa_continuous(ETA, LAM, SIGMA)
    tau = 1e-5
    kappa_t = kappa_tilde_discrete(tau, ETA, LAM, SIGMA)
    assert abs(kappa_t - kappa_c) < 1e-6


def test_boundary_conditions():
    """x(0) = x0 and x(T) = 0 exactly (full liquidation)."""
    x0_val = x_continuous(torch.tensor([0.0]), X0, T, ETA, LAM, SIGMA).item()
    xT_val = x_continuous(torch.tensor([T]), X0, T, ETA, LAM, SIGMA).item()
    assert math.isclose(x0_val, X0, rel_tol=1e-9)
    assert abs(xT_val) < 1e-9


def test_trajectory_monotonically_decreasing():
    """Optimal liquidation never buys back: x(t) should be non-increasing."""
    t_grid = torch.linspace(0, T, 200, dtype=torch.float64)
    x_path = x_continuous(t_grid, X0, T, ETA, LAM, SIGMA)
    diffs = x_path[1:] - x_path[:-1]
    assert (diffs <= 1e-12).all()


def test_trading_rate_matches_finite_difference():
    """v_continuous (autograd) should match a central finite difference of x_continuous."""
    t0 = 0.5
    eps = 1e-4
    x_plus = x_continuous(torch.tensor([t0 + eps], dtype=torch.float64), X0, T, ETA, LAM, SIGMA).item()
    x_minus = x_continuous(torch.tensor([t0 - eps], dtype=torch.float64), X0, T, ETA, LAM, SIGMA).item()
    fd_deriv = (x_plus - x_minus) / (2 * eps)
    v_auto = v_continuous(torch.tensor([t0], dtype=torch.float64), X0, T, ETA, LAM, SIGMA).item()
    assert math.isclose(v_auto, -fd_deriv, rel_tol=1e-5)


def test_hjb_residual_near_zero_for_true_solution():
    """The true h(t) should satisfy the HJB/Riccati equation to ~machine precision."""
    t_test = torch.tensor([0.1, 0.3, 0.5, 0.7, 0.9], dtype=torch.float64)
    x_test = torch.tensor([0.8, 0.6, 0.5, 0.3, 0.1], dtype=torch.float64)
    R = hjb_residual(t_test, x_test, T, ETA, LAM, SIGMA)
    assert R.abs().max().item() < 1e-10


def test_hjb_residual_blows_up_when_perturbed():
    """A perturbed (scaled) h(t) should NOT satisfy the HJB equation."""
    t_test = torch.tensor([0.1, 0.3, 0.5, 0.7, 0.9], dtype=torch.float64)
    x_test = torch.tensor([0.8, 0.6, 0.5, 0.3, 0.1], dtype=torch.float64)
    R_true = hjb_residual(t_test, x_test, T, ETA, LAM, SIGMA)
    R_bad = hjb_residual_perturbed(t_test, x_test, T, ETA, LAM, SIGMA, scale=1.2)
    assert R_bad.abs().max().item() > 100 * max(R_true.abs().max().item(), 1e-12)


@pytest.mark.parametrize("N1,N2", [(10, 100), (100, 1000)])
def test_discrete_converges_to_continuous(N1, N2):
    """
    Doubling-style check: the discrete-vs-continuous error at finer N2
    should be smaller than at coarser N1, and roughly consistent with the
    expected O(tau^2) rate (error ratio ~ (tau1/tau2)^2, with some slack).
    """
    def max_err(N):
        j = torch.arange(0, N + 1, dtype=torch.float64)
        t_disc = j * (T / N)
        x_disc = x_discrete(j, X0, N, T, ETA, LAM, SIGMA)
        x_cont = x_continuous(t_disc, X0, T, ETA, LAM, SIGMA)
        return (x_disc - x_cont).abs().max().item()

    err1 = max_err(N1)
    err2 = max_err(N2)
    assert err2 < err1

    tau1, tau2 = T / N1, T / N2
    expected_ratio = (tau1 / tau2) ** 2
    actual_ratio = err1 / err2
    # allow generous slack since this is an asymptotic rate, not exact
    assert 0.3 * expected_ratio < actual_ratio < 3.0 * expected_ratio


def test_reproducibility_fixed_seed():
    """Closed-form functions are deterministic given fixed inputs (no RNG
    involved here, but this documents the seed-discipline the course asks
    for and gives a regression check on exact values)."""
    torch.manual_seed(0)
    val_a = x_continuous(torch.tensor([0.37]), X0, T, ETA, LAM, SIGMA).item()
    torch.manual_seed(0)
    val_b = x_continuous(torch.tensor([0.37]), X0, T, ETA, LAM, SIGMA).item()
    assert val_a == val_b